# 02 Parameter Ensemble

Equal voting across 6,336 parameter combinations produces daily ETF weights. Seven decision-tree parameters take several plausible values; combinations are retained only when `extreme > fear`. Each valid set selects one ETF per day. Its vote contributes 1/6,336 to that asset's target weight.

Signals are generated at the close of T and executed at the open of T+1. The portfolio is compared with the original single-parameter baseline and SPY/TQQQ buy-and-hold benchmarks. This experiment studies parameter voting without adding the legacy stop-loss/take-profit grid.

The notebook can be run independently. The current one-click workflow also calls it before V1 and V2 allocation. Its historical filename was `09_parameter_ensemble_2.ipynb`.


**Submission reference:** Quote historical final-portfolio performance and comparisons from the `Submission reference: historical IS / OOS metrics` table in `results/Report.html` (`results/tables/final_metrics.csv`, generated by 07). Identify the strategy row and period. This notebook's standalone performance tables are research diagnostics; their figures must not be mixed with the final account metrics.


In [ ]:
from pathlib import Path
import os
import json
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_pipeline_dir():
    """Supports execution from the project root, pipeline directory and project subdirectories."""
    roots = (
        [Path(os.environ["ETF_PROJECT_ROOT"]).expanduser()]
        if os.environ.get("ETF_PROJECT_ROOT")
        else []
    )
    roots += [Path.cwd(), *Path.cwd().parents]

    for root in roots:
        for candidate in (root / "pipeline", root):
            if (candidate / "00_engine.ipynb").is_file():
                return candidate.resolve()

    raise FileNotFoundError(
        "Cannot find pipeline Directory.Set the working directory to the project root or pipeline Directory."
    )


PIPELINE_DIR = find_pipeline_dir()


def load_notebook_definitions(filename):
    """Load notebook definitions in; Skip cells tagged pipeline-entry tag entry cell."""
    path = next(iter(sorted(PIPELINE_DIR.rglob(filename))), PIPELINE_DIR / filename)   # At pipeline Search subdirectories
    notebook = json.loads(path.read_text(encoding="utf-8"))
    namespace = {"__name__": "pipeline_module"}

    for number, item in enumerate(notebook["cells"]):
        if item["cell_type"] != "code":
            continue
        if "pipeline-entry" in item.get("metadata", {}).get("tags", []):
            continue

        exec(
            compile("".join(item["source"]), f"{path}:cell{number}", "exec"),
            namespace,
        )

    return namespace


# ------------------------------------------------------------
# Bind explicitly 00_engine.ipynb objects used by this file.
# No longer used globals()[name] Inject dynamically, In this way PyCharm Recognizes these names, 
# This removes IS_START / TICKS / prep / backtest etc. unresolved reference Red line.
# ------------------------------------------------------------
_engine = load_notebook_definitions("00_engine.ipynb")

rebalance_exact = _engine["rebalance_exact"]
IS_START = _engine["IS_START"]
IS_END = _engine["IS_END"]
OOS_START = _engine["OOS_START"]
OOS_END = _engine["OOS_END"]
TICKS = _engine["TICKS"]
ONE_WAY_BPS = _engine["ONE_WAY_BPS"]

wilder_rsi = _engine["wilder_rsi"]
prep = _engine["prep"]
signals = _engine["signals"]
backtest = _engine["backtest"]
metrics = _engine["metrics"]
fmt = _engine["fmt"]


## 1. Parameter grid

The grid maps onto the original decision tree as follows:
- RSI lookback: `rsi_n`.
- SPY moving average: `spy_sma`.
- Overbought threshold: both `qqq_rsi_bull` and `spy_rsi_bull`.
- Oversold threshold: `tqqq_rsi_dip` and the original fixed `SPY RSI < 30` test.
- UVXY panic threshold: `uvxy_hi`.
- UVXY extreme threshold: `uvxy_ext`.
- SQQQ threshold: `sqqq_exit` and the original fixed `SQQQ RSI < 31` test.

QQQ and TQQQ retain their 20-day moving averages.


In [ ]:
PARAM_GRID = {
    'rsi_n':[7,10,14],
    'spy_sma':[100,120,150,200],
    'overheat':[70,75,80,85],
    'oversold':[20,25,30,35],
    'uvxy_fear':[64,70,74,80],
    'uvxy_extreme':[80,84,88],
    'sqqq_line':[30,34,38],
}

def build_parameter_combinations():
    keys=list(PARAM_GRID)
    combos=[]
    for vals in itertools.product(*(PARAM_GRID[k] for k in keys)):
        p=dict(zip(keys,vals))
        if p['uvxy_extreme'] > p['uvxy_fear']:
            combos.append(p)
    assert len(combos)==6336, f'Should be6336sets, Actual{len(combos)}'
    return combos

PARAM_COMBOS=build_parameter_combinations()
print('Number of valid parameter combinations:', len(PARAM_COMBOS))
display(pd.DataFrame(PARAM_COMBOS).head())


## 2. Precompute RSI and moving averages

Compute RSI at 7/10/14 days, SPY SMA at 100/120/150/200 days, and QQQ/TQQQ SMA20 once, avoiding repeated indicator calculations across parameter sets.


In [ ]:
def precompute_indicators(C):
    """Convert all to NumPy, Avoid 6336×Repeated operations in a loop over thousands of days pandas iloc."""
    ticker_to_col={t:i for i,t in enumerate(TICKS)}
    rsi_cache={}
    for n in PARAM_GRID['rsi_n']:
        rsi_cache[n]=np.column_stack([
            wilder_rsi(C[t].values,n) for t in TICKS
        ])
    return {
        'ticker_to_col':ticker_to_col,
        'rsi':rsi_cache,
        'spy_sma':{n:C['SPY'].rolling(n).mean().to_numpy(float) for n in PARAM_GRID['spy_sma']},
        'qqq_sma20':C['QQQ'].rolling(20).mean().to_numpy(float),
        'tqqq_sma20':C['TQQQ'].rolling(20).mean().to_numpy(float),
        'prices':{t:C[t].to_numpy(float) for t in ['SPY','QQQ','TQQQ']},
    }


## 3. Select one ETF per parameter set each day

Retain the `signals()` decision tree from `00_engine.ipynb`, allowing the seven grid parameters to vary.


In [ ]:
def targets_for_parameter_set(C, indicators, p):
    """Generate the full historical ETF target codes for one parameter set ETF target code; Compared with daily pandas Much faster than daily comparisons."""
    col=indicators['ticker_to_col']
    r=indicators['rsi'][p['rsi_n']]
    s_spy=indicators['spy_sma'][p['spy_sma']]
    s_qqq=indicators['qqq_sma20']
    s_tqqq=indicators['tqqq_sma20']
    pspy=indicators['prices']['SPY']
    pqqq=indicators['prices']['QQQ']
    ptq=indicators['prices']['TQQQ']

    n=len(C)
    target=np.full(n,-1,dtype=np.int16)
    valid=(~np.isnan(r).any(axis=1) & ~np.isnan(s_spy) & ~np.isnan(s_qqq) & ~np.isnan(s_tqqq))

    # Bull branch
    bull=valid & (pspy>s_spy)
    hot=bull & ((r[:,col['QQQ']]>p['overheat']) | (r[:,col['SPY']]>p['overheat']))
    target[bull]=col['TQQQ']
    target[hot]=col['UVXY']

    # Bear branch, Preserve the original exactly if/elif Order
    rem=valid & ~bull

    m=rem & (r[:,col['TQQQ']] < p['oversold'])
    target[m]=col['TECL']; rem &= ~m

    m=rem & (r[:,col['SPY']] < p['oversold'])
    target[m]=col['SPXL']; rem &= ~m

    fear=rem & (r[:,col['UVXY']] > p['uvxy_fear'])
    extreme=fear & (r[:,col['UVXY']] > p['uvxy_extreme'])
    non_extreme=fear & ~extreme
    target[non_extreme]=col['UVXY']

    ext_q_up=extreme & (pqqq>s_qqq)
    target[ext_q_up & (r[:,col['SQQQ']] < p['sqqq_line'])]=col['TECS']
    target[ext_q_up & ~(r[:,col['SQQQ']] < p['sqqq_line'])]=col['TECL']

    ext_q_down=extreme & ~(pqqq>s_qqq)
    target[ext_q_down & (r[:,col['TECS']] >= r[:,col['BSV']])]=col['TECS']
    target[ext_q_down & (r[:,col['TECS']] < r[:,col['BSV']])]=col['BSV']

    # Non-panic branch
    normal=rem & ~fear
    tq_up=normal & (ptq>s_tqqq)
    target[tq_up & (r[:,col['SQQQ']] < p['sqqq_line'])]=col['TECS']
    target[tq_up & ~(r[:,col['SQQQ']] < p['sqqq_line'])]=col['TECL']

    tq_down=normal & ~(ptq>s_tqqq)
    target[tq_down & (r[:,col['TECS']] >= r[:,col['BSV']])]=col['TECS']
    target[tq_down & (r[:,col['TECS']] < r[:,col['BSV']])]=col['BSV']

    return target,valid


## 4. Convert 6,336 votes into daily ETF weights

The formal sample begins at `IS_START`. With a maximum moving average of 200 days, every formal sample date must have all 6,336 valid votes.


In [ ]:
def generate_ensemble_weights(C, combos=None, start=None, verbose=True):
    combos=PARAM_COMBOS if combos is None else combos
    indicators=precompute_indicators(C)
    n=len(C); m=len(TICKS)
    votes_np=np.zeros((n,m),dtype=np.int32)
    valid_n_np=np.zeros(n,dtype=np.int32)
    start_ts=pd.Timestamp(start) if start is not None else None
    start_pos=0 if start_ts is None else C.index.searchsorted(start_ts)
    rows=np.arange(n)

    for j,p in enumerate(combos,1):
        target,valid=targets_for_parameter_set(C,indicators,p)
        if start_pos>0:
            valid[:start_pos]=False
        idx=rows[valid]
        np.add.at(votes_np,(idx,target[valid]),1)
        valid_n_np[valid]+=1
        if verbose and (j%1000==0 or j==len(combos)):
            print(f'Completed {j}/{len(combos)} parameter sets')

    votes=pd.DataFrame(votes_np,index=C.index,columns=TICKS)
    valid_n=pd.Series(valid_n_np,index=C.index,name='valid_parameter_count')
    weights=votes.div(valid_n.replace(0,np.nan),axis=0)

    formal=valid_n.loc[pd.Timestamp(start or IS_START):]
    formal=formal[formal>0]
    if len(formal) and (formal!=len(combos)).any():
        bad=formal[formal!=len(combos)]
        raise ValueError(f'Fewer than required combinations in the formal backtest period6336group dates, For example: {bad.head().to_dict()}')

    # Basic consistency checks
    chk=weights.loc[formal.index].sum(axis=1)
    if len(chk) and not np.allclose(chk.values,1.0,atol=1e-10):
        raise AssertionError('Some dates ETF Weights do not sum to 1.')

    return weights,votes,valid_n


## 5. Multi-asset portfolio backtest

The original `backtest()` holds one ETF at a time. This portfolio backtester supports multiple weighted positions. Signals formed at the close of T execute at the open of T+1. Per-asset one-way fee rates and the exact transaction-cost solver are supplied by `00_engine.ipynb`.


In [ ]:
def backtest_weighted_portfolio(O,C,weights,start=IS_START):
    idx=C.index; tickers=list(TICKS)
    qty=np.zeros(len(tickers),dtype=float)
    eq=pd.Series(np.nan,index=idx,dtype=float)
    turnover=pd.Series(np.nan,index=idx,dtype=float)
    cost_paid=pd.Series(0.0,index=idx,dtype=float)
    nav=1.0; pending=None; started=False

    for d in idx:
        if d < pd.Timestamp(start):
            continue
        open_px=O.loc[d,tickers].to_numpy(float)
        close_px=C.loc[d,tickers].to_numpy(float)

        if not started:
            started=True
            eq.loc[d]=nav
        else:
            current_value=float(np.dot(qty,open_px)) if qty.sum()!=0 else nav
            if pending is not None:
                new_w=pending.reindex(tickers).fillna(0.0).to_numpy(float)
                if qty.sum()==0:
                    old_w=np.zeros(len(tickers))
                else:
                    av=qty*open_px
                    old_w=av/av.sum()
                delta=new_w-old_w
                rates=np.array([ONE_WAY_BPS[t]/1e4 for t in tickers])
                dollars, fees = rebalance_exact(current_value, qty*open_px, new_w, rates)
                tc=float(fees.sum())
                investable=max(current_value-tc,0.0)
                qty=np.divide(investable*new_w,open_px,out=np.zeros_like(new_w),where=open_px>0)
                turnover.loc[d]=float(np.abs(dollars).sum()/current_value)
                cost_paid.loc[d]=tc
            nav=float(np.dot(qty,close_px))
            eq.loc[d]=nav

        row=weights.loc[d]
        if row.notna().any():
            pending=row.fillna(0.0)

    eq=eq.dropna()
    diagnostics=pd.DataFrame({'turnover':turnover,'cost_paid':cost_paid}).loc[eq.index]
    return eq,diagnostics


## 6. Compare with the original baseline SPY and TQQQ


In [ ]:
def buy_and_hold_equity(C,ticker,start=IS_START):
    s=C[ticker].loc[start:].dropna()
    return s/s.iloc[0]

def evaluate_results(C,ensemble_eq,original_eq):
    rows=[]
    series_map={
        'Ensemble 6336':ensemble_eq,
        'Original baseline':original_eq,
        'SPY Buy&Hold':buy_and_hold_equity(C,'SPY'),
        'TQQQ Buy&Hold':buy_and_hold_equity(C,'TQQQ'),
    }
    for name,eq in series_map.items():
        for label,a,b in [('IS',IS_START,IS_END),('OOS',OOS_START,OOS_END),('OOS ex-2020','2021-01-01',OOS_END)]:
            if len(eq.loc[a:b])<2:
                continue
            rows.append({'Strategy':name,'Period':label,**fmt(metrics(eq,a=a,b=b))})
    table=pd.DataFrame(rows)
    display(table)
    return table

def plot_results(weights,ensemble_eq,original_eq,C):
    fig1,ax1=plt.subplots(figsize=(11,5))
    weights.loc[IS_START:].plot.area(ax=ax1,stacked=True)
    ax1.set_title('Parameter Ensemble ETF Weights')
    ax1.set_ylabel('Portfolio weight'); ax1.set_ylim(0,1)
    ax1.legend(loc='upper left',bbox_to_anchor=(1.01,1))
    plt.tight_layout(); plt.show()

    fig2,ax2=plt.subplots(figsize=(11,5))
    (ensemble_eq/ensemble_eq.iloc[0]).plot(ax=ax2,label='Ensemble 6336')
    (original_eq/original_eq.iloc[0]).plot(ax=ax2,label='Original baseline')
    buy_and_hold_equity(C,'SPY',ensemble_eq.index[0]).plot(ax=ax2,label='SPY Buy&Hold')
    buy_and_hold_equity(C,'TQQQ',ensemble_eq.index[0]).plot(ax=ax2,label='TQQQ Buy&Hold')
    ax2.axvline(pd.Timestamp(OOS_START),color='gray',ls='--')
    ax2.set_yscale('log'); ax2.set_title('Equity Curves (log scale)'); ax2.legend()
    plt.tight_layout(); plt.show()
    return fig1,fig2


## 7. Run the notebook

Restart the kernel and select Run All, or use the root `run_all.py` workflow.


In [ ]:
def main(csv_path=None,start=IS_START,verbose=True):
    O,H,L,C=prep(csv_path)
    print('Data range:',C.index[0].date(),'~',C.index[-1].date())
    print('Number of parameter combinations:',len(PARAM_COMBOS))

    weights,votes,valid_n=generate_ensemble_weights(C,PARAM_COMBOS,start,verbose)
    ensemble_eq,diag=backtest_weighted_portfolio(O,C,weights,start=start)

    original_tg=signals(C)
    original_eq,original_trades=backtest(O,H,L,C,original_tg,None,None,True,start=start)

    table=evaluate_results(C,ensemble_eq,original_eq)
    latest_date=weights.dropna(how='all').index[-1]
    latest=pd.DataFrame({
        'votes':votes.loc[latest_date],
        'weight':weights.loc[latest_date],
        'weight_%':weights.loc[latest_date]*100,
    }).sort_values('weight',ascending=False)
    print('\nLatest ensemble date:',latest_date.date())
    display(latest.round({'weight':4,'weight_%':2}))
    figs=plot_results(weights,ensemble_eq,original_eq,C)
     # ---- Save to results/ ----
    root=PIPELINE_DIR.parent if PIPELINE_DIR.name=='pipeline' else PIPELINE_DIR
    tdir,fdir,sdir=root/'results'/'tables',root/'results'/'figures',root/'results'/'signals'
    for d in (tdir,fdir,sdir): d.mkdir(parents=True,exist_ok=True)
    table.to_csv(tdir/'ensemble_metrics.csv',index=False,encoding='utf-8-sig')
    diag.to_csv(tdir/'ensemble_turnover_cost.csv',encoding='utf-8-sig')
    weights.dropna(how='all').to_csv(sdir/'ensemble_weights_history.csv',encoding='utf-8-sig')
    latest.to_csv(sdir/'ensemble_latest_allocation.csv',encoding='utf-8-sig')
    figs[0].savefig(fdir/'ensemble_weights.png',dpi=150,bbox_inches='tight')
    figs[1].savefig(fdir/'ensemble_equity.png',dpi=150,bbox_inches='tight')
    print('Saved to',root/'results')
    print('\nParameter ensemble Run complete.')

    return {
        'parameter_combinations':PARAM_COMBOS,
        'weights':weights,
        'votes':votes,
        'valid_parameter_count':valid_n,
        'ensemble_equity':ensemble_eq,
        'ensemble_diagnostics':diag,
        'original_baseline_equity':original_eq,
        'metrics':table,
        'latest_date':latest_date,
        'latest_allocation':latest,
        'figures':figs,
    }


In [ ]:
# ===== Only this section needs modification, Then Run All =====
CSV_PATH=None
START=IS_START
VERBOSE=True

results_ensemble=main(
    csv_path=CSV_PATH,
    start=START,
    verbose=VERBOSE,
)
